In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from dotenv import load_dotenv, find_dotenv
from utils.outlier_detection import detect_outliers_by_series
from utils.formatting_functions import *
from utils.db_functions import *
from utils.excel_functions import *

In [10]:
# Read from gdrive or locally
use_gdrive = False

# Data path
if use_gdrive:
    gdrive_path = os.path.join(
        find_shared_folder("ISPAR Team Shared"),"Metadata Automation",)
    data_path = Path(gdrive_path) / 'Data'
else:
    data_path = Path('C:/Users/alfon/Desktop/UN/Data')

metadata_output_path = Path(data_path) / 'Metadata Automation Outputs'

In [11]:
import os
from collections import defaultdict
from pathlib import Path

import pandas as pd

SHEET = 'Summary Stats'

# ---------------------------------------------------------------------------
# 1. Catalogue files and read 'Summary Stats'
# ---------------------------------------------------------------------------
files = sorted(
    p for p in metadata_output_path.glob('*.xlsx')
    if not p.name.startswith('~$')  # skip Excel temp/lock files
)
print(f"Found {len(files)} Excel files in {metadata_output_path}\n")

records = []        # one row per file
stats = {}          # file name -> Summary Stats dataframe
sheet_names = {}    # file name -> list of sheet names

for f in files:
    rec = {'file': f.name, 'sheets': None, 'categories': None, 'error': None}
    try:
        xl = pd.ExcelFile(f)
        sheet_names[f.name] = xl.sheet_names
        rec['sheets'] = tuple(xl.sheet_names)
        if SHEET not in xl.sheet_names:
            rec['error'] = f"Sheet '{SHEET}' not found"
        else:
            df = xl.parse(SHEET)
            df.columns = [str(c).strip() for c in df.columns]
            cat_col = df.columns[0]  # first column = Category
            df[cat_col] = df[cat_col].astype(str).str.strip()
            stats[f.name] = df.set_index(cat_col)
            rec['categories'] = tuple(df[cat_col].tolist())
    except Exception as e:
        rec['error'] = repr(e)
    records.append(rec)

catalogue = pd.DataFrame(records)

# ---------------------------------------------------------------------------
# 2. Check sheet names are consistent across files
# ---------------------------------------------------------------------------
print("=" * 70)
print("SHEET NAME CHECK")
print("=" * 70)
sheet_groups = defaultdict(list)
for name, sheets in sheet_names.items():
    sheet_groups[tuple(sorted(sheets))].append(name)
if len(sheet_groups) <= 1:
    print("All files have the same sheet names.\n")
else:
    for i, (sheets, names) in enumerate(sheet_groups.items(), 1):
        print(f"Sheet group {i}: {list(sheets)}")
        for n in names:
            print(f"   - {n}")
    print()

# ---------------------------------------------------------------------------
# 3. Group files by identical category set (order-insensitive)
# ---------------------------------------------------------------------------
print("=" * 70)
print("CATEGORY GROUPS (files with identical categories)")
print("=" * 70)
groups = defaultdict(list)
for rec in records:
    if rec['categories'] is not None:
        groups[frozenset(rec['categories'])].append(rec['file'])

all_cats = set().union(*groups.keys()) if groups else set()
common_cats = set.intersection(*[set(k) for k in groups.keys()]) if groups else set()

for i, (cats, names) in enumerate(
        sorted(groups.items(), key=lambda kv: -len(kv[1])), 1):
    print(f"\nGroup {i}: {len(names)} file(s), {len(cats)} categories")
    print(f"  Categories: {sorted(cats)}")
    print(f"  Missing vs. union of all files: {sorted(all_cats - cats) or 'none'}")
    print(f"  Extra vs. common to all files:  {sorted(cats - common_cats) or 'none'}")
    for n in names:
        print(f"    - {n}")

# ---------------------------------------------------------------------------
# 4. Per-file presence matrix (which file has which category)
# ---------------------------------------------------------------------------
matrix = pd.DataFrame(
    {name: {c: (c in df.index) for c in sorted(all_cats)}
     for name, df in stats.items()}
).T
matrix.index.name = 'file'

print("\n" + "=" * 70)
print("CATEGORY PRESENCE MATRIX (True = file has the category)")
print("=" * 70)
print(matrix.to_string())

# Files with problems
bad = catalogue[catalogue['error'].notna()]
if not bad.empty:
    print("\n" + "=" * 70)
    print("FILES WITH PROBLEMS")
    print("=" * 70)
    print(bad[['file', 'error']].to_string(index=False))

# ---------------------------------------------------------------------------
# 5. Save results next to the data
# ---------------------------------------------------------------------------
out_xlsx = data_path /'Metadata Automation Audit'/ 'summary_stats_category_comparison.xlsx'

with pd.ExcelWriter(out_xlsx) as writer:
    catalogue.assign(
        sheets=catalogue['sheets'].astype(str),
        categories=catalogue['categories'].astype(str),
    ).to_excel(writer, sheet_name='Catalogue', index=False)
    matrix.to_excel(writer, sheet_name='Category Matrix')
    pd.DataFrame(
        [(i, len(n), sorted(c), n) for i, (c, n) in enumerate(groups.items(), 1)],
        columns=['group', 'n_files', 'categories', 'files'],
    ).astype(str).to_excel(writer, sheet_name='Groups', index=False)
print(f"\nSaved comparison to {out_xlsx}")

Found 24 Excel files in C:\Users\alfon\Desktop\UN\Data\Metadata Automation Outputs

SHEET NAME CHECK
All files have the same sheet names.

CATEGORY GROUPS (files with identical categories)

Group 1: 23 file(s), 9 categories
  Categories: ['A1_cap_adjusted_match', 'A_total', 'B_total', 'C_data_available_no_match', 'D_possible_impute', 'E_no_historical_data', 'F_no_report_data', 'G_no_historical_and_report_data', 'Total']
  Missing vs. union of all files: none
  Extra vs. common to all files:  ['G_no_historical_and_report_data']
    - Metadata_Automation_index_10.xlsx
    - Metadata_Automation_index_1021.xlsx
    - Metadata_Automation_index_1022.xlsx
    - Metadata_Automation_index_1024.xlsx
    - Metadata_Automation_index_1026.xlsx
    - Metadata_Automation_index_1027.xlsx
    - Metadata_Automation_index_1030.xlsx
    - Metadata_Automation_index_1031.xlsx
    - Metadata_Automation_index_1032.xlsx
    - Metadata_Automation_index_1034.xlsx
    - Metadata_Automation_index_1035.xlsx
    - M